# Study 0 — SSL Pretraining Experiment (ATS-167)

**Question:** Does self-supervised pretraining (masked feature reconstruction) improve the FT-Transformer's distress prediction performance compared to training from scratch?

**Approach:**
- Sweep mask ratios: 15%, 30%, 50%
- Pretrain 200 epochs (linear warmup + cosine annealing)
- Fine-tune on all 5 distress outcomes
- Compare AUROC vs from-scratch baseline

**Decision rule:** If gains are marginal, move on — the primary question is FT-Transformer vs XGBoost.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from fin_jepa.training.pretrain_ssl import run_ssl_experiment

sns.set_theme(style="whitegrid", palette="colorblind")
RESULTS_PATH = Path("../../results/study0/ssl_experiment_results.json")

## 1. Run Experiment

In [ ]:
# Load from disk if already run, otherwise execute the experiment.
if RESULTS_PATH.exists():
    print(f"Loading cached results from {RESULTS_PATH}")
    with open(RESULTS_PATH) as f:
        results = json.load(f)
else:
    print("Running SSL experiment — this will take a while on CPU...")
    config = {
        "data": {
            "raw_dir": "../../data/raw",
            "processed_dir": "../../data/processed",
            "split": {
                "train_end": "2017-12-31",
                "val_end": "2019-12-31",
                "test_end": "2023-12-31",
            },
        },
        "features": {
            "use_raw": True,
            "use_ratios": True,
            "use_yoy": True,
            "use_missingness_flags": True,
            "coverage_threshold": 0.50,
            "normalization_method": "quantile",
            "median_impute": True,
        },
        "outcomes": [
            "stock_decline",
            "earnings_restate",
            "audit_qualification",
            "sec_enforcement",
            "bankruptcy",
        ],
        "ssl_experiment": {"mask_ratios": [0.15, 0.30, 0.50]},
        "pretrain": {
            "batch_size": 512,
            "epochs": 200,
            "learning_rate": 1e-4,
            "weight_decay": 1e-5,
            "warmup_epochs": 10,
        },
        "ft_transformer": {
            "d_token": 192,
            "n_heads": 8,
            "n_layers": 3,
            "d_ffn_factor": 4,
            "dropout": 0.0,
        },
        "training": {"seed": 42},
        "checkpoint_dir": "../../models/checkpoints/study0_ssl_experiment",
        "results_dir": "../../results/study0",
    }
    results = run_ssl_experiment(config)
    print("Done.")

print(f"Recommendation: {results['recommendation']}")

## 2. Results Summary

In [ ]:
# Build comparison table: outcome x (baseline, pretrained mask ratios, delta)
outcomes = list(results["baseline"].keys())
mask_ratios = sorted(results["pretrained"].keys())

rows = []
for oc in outcomes:
    row = {"Outcome": oc}
    bl = results["baseline"][oc].get("auroc")
    row["Baseline AUROC"] = bl

    for mr in mask_ratios:
        pt_auroc = results["pretrained"][mr][oc].get("auroc")
        row[f"MR={mr}"] = pt_auroc

    comp = results["comparison"].get(oc, {})
    row["Best Delta"] = comp.get("delta")
    row["Best MR"] = comp.get("best_mask_ratio")
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index("Outcome")

def highlight_positive(val):
    if isinstance(val, (int, float)) and val > 0:
        return "color: green; font-weight: bold"
    elif isinstance(val, (int, float)) and val < 0:
        return "color: red"
    return ""

summary_df.style.map(highlight_positive, subset=["Best Delta"]).format(
    {c: "{:.4f}" for c in summary_df.columns if c != "Best MR"}, na_rep="—"
)

## 3. Pretrained vs From-Scratch Comparison

In [ ]:
# Grouped bar chart: AUROC by outcome for baseline + each mask ratio
plot_data = []
for oc in outcomes:
    bl = results["baseline"][oc].get("auroc")
    if bl is not None:
        plot_data.append({"Outcome": oc, "Variant": "Baseline", "AUROC": bl})
    for mr in mask_ratios:
        pt = results["pretrained"][mr][oc].get("auroc")
        if pt is not None:
            plot_data.append({"Outcome": oc, "Variant": f"MR={mr}", "AUROC": pt})

plot_df = pd.DataFrame(plot_data)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=plot_df, x="Outcome", y="AUROC", hue="Variant", ax=ax)
ax.set_title("AUROC: From-Scratch vs SSL-Pretrained (by mask ratio)")
ax.set_ylim(0.4, 1.0)
ax.legend(title="Variant", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 4. Pretraining Loss Curves

In [ ]:
# Reconstruction loss over pretraining epochs for each mask ratio
fig, ax = plt.subplots(figsize=(10, 4))
for mr in sorted(results["loss_curves"].keys()):
    losses = results["loss_curves"][mr]
    ax.plot(range(1, len(losses) + 1), losses, label=f"MR={mr}")

ax.set_xlabel("Epoch")
ax.set_ylabel("Reconstruction Loss (MSE)")
ax.set_title("SSL Pretraining Loss Curves")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Per-Outcome Detailed Metrics

In [ ]:
# Detailed metrics table for each outcome
metrics_of_interest = ["auroc", "auprc", "brier", "f1_youden"]

for oc in outcomes:
    rows = []
    bl = results["baseline"][oc]
    rows.append({"Variant": "Baseline", **{m: bl.get(m) for m in metrics_of_interest}})
    for mr in mask_ratios:
        pt = results["pretrained"][mr][oc]
        rows.append({"Variant": f"MR={mr}", **{m: pt.get(m) for m in metrics_of_interest}})

    df = pd.DataFrame(rows).set_index("Variant")
    print(f"\n{'='*60}")
    print(f"  {oc}")
    print(f"{'='*60}")
    display(df.style.format("{:.4f}", na_rep="—"))

## 6. Decision

In [ ]:
rec = results["recommendation"]
n_improved = sum(
    1 for c in results["comparison"].values()
    if not c.get("skipped") and c.get("delta", 0) >= 0.005
)

print(f"Recommendation: {rec.upper()}")
print(f"Outcomes improved (delta >= 0.005): {n_improved}/5\n")

if rec == "proceed":
    print("SSL pretraining shows meaningful gains on >= 3/5 outcomes.")
    print("Use pretrained initialisation as default for Studies 1-2.")
elif rec == "marginal":
    print("SSL pretraining shows marginal gains (1-2 outcomes).")
    print("Consider keeping as optional; primary gate remains FT vs XGBoost.")
else:
    print("SSL pretraining shows no meaningful gains.")
    print("Move on — the primary question is FT-Transformer vs XGBoost.")

print("\nPer-outcome deltas:")
for oc, comp in results["comparison"].items():
    if comp.get("skipped"):
        print(f"  {oc}: skipped")
    else:
        d = comp["delta"]
        sign = "+" if d >= 0 else ""
        print(f"  {oc}: {sign}{d:.4f} (best MR={comp['best_mask_ratio']})")